<a href="https://colab.research.google.com/github/hassnainsvtti-collab/AI-with-SVTTI/blob/main/Lecture_05_File_Handling_Exceptions_MiniProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 2 — Python for AI Engineering
## Lecture 5: File Handling, Exception Handling + Module 2 Mini Project
**Dev Valley Software House & Incubation Centre, Gujranwala**

---
> **In this lecture you will learn:**
> - Reading and writing files (text, CSV, JSON)
> - Exception handling — writing robust, crash-proof code
> - Python context managers (`with` statement)
> - Module 2 full recap
> - **Mini Project:** End-to-end Student Data Analysis Pipeline
> - Preview of Module 3 — Machine Learning

---

## 1. File Handling

AI projects constantly read and write files:
- Loading datasets (`.csv`, `.json`, `.txt`)
- Saving model weights
- Logging training progress
- Writing results and reports

### The `with` statement (context manager)
Always use `with open(...) as f:` — it automatically closes the file, even if an error occurs.

```python
# Old way — easy to forget to close
f = open("file.txt", "r")
data = f.read()
f.close()          # <-- easy to forget!

# Better way — closes automatically
with open("file.txt", "r") as f:
    data = f.read()
```

### File modes
| Mode | Meaning |
|------|---------|
| `'r'` | Read (default) |
| `'w'` | Write (overwrites existing file) |
| `'a'` | Append (adds to end of file) |
| `'r+'`| Read and write |

In [ ]:
# ── CODE CELL 1 ──────────────────────────────────────────────────────────────
# Text file — write, read, and append

filepath = "/content/training_log.txt"

# Write initial content
with open(filepath, "w") as f:
    f.write("=== Model Training Log ===\n")
    f.write("Course: AI Engineering — Dev Valley\n")
    f.write("Date: 2024-01-15\n\n")

# Append epoch results
results = [
    (1, 0.95, 0.98),
    (2, 0.80, 0.84),
    (3, 0.68, 0.74),
    (4, 0.58, 0.66),
    (5, 0.50, 0.60),
]
with open(filepath, "a") as f:
    f.write(f"{'Epoch':<8} {'Train Loss':<14} {'Val Loss'}\n")
    f.write("-" * 35 + "\n")
    for epoch, train, val in results:
        f.write(f"{epoch:<8} {train:<14.3f} {val:.3f}\n")

# Read and print the file
with open(filepath, "r") as f:
    content = f.read()

print(content)

In [ ]:
# ── CODE CELL 2 ──────────────────────────────────────────────────────────────
# JSON files — storing dictionaries and model configs
# JSON is the standard format for AI model configuration

import json

# A typical AI model configuration
model_config = {
    "model_name":     "StudentMarksPredictor",
    "version":        "1.0",
    "architecture":   "Linear Regression",
    "hyperparameters":{
        "learning_rate": 0.01,
        "epochs":        100,
        "batch_size":    32,
    },
    "features":       ["study_hours", "sleep_hours", "age"],
    "target":         "marks",
    "metrics":        {"train_loss": 0.23, "val_loss": 0.27, "r2_score": 0.89},
}

# Save to JSON
with open("/content/model_config.json", "w") as f:
    json.dump(model_config, f, indent=4)
print("Config saved to model_config.json")

# Load it back
with open("/content/model_config.json", "r") as f:
    loaded_config = json.load(f)

print("\nLoaded config:")
print(f"  Model   : {loaded_config['model_name']} v{loaded_config['version']}")
print(f"  Features: {loaded_config['features']}")
print(f"  R2 Score: {loaded_config['metrics']['r2_score']}")

## 2. Exception Handling

AI code runs for hours — a single unhandled error crashes everything. Exception handling lets your code **fail gracefully and recover**.

### Syntax
```python
try:
    # Code that might fail
    result = 10 / 0
except ZeroDivisionError as e:
    # Handle the specific error
    print(f"Error: {e}")
except Exception as e:
    # Catch any other error
    print(f"Unexpected error: {e}")
else:
    # Runs only if NO exception occurred
    print("Success!")
finally:
    # Always runs — good for cleanup
    print("Done.")
```

### Common exceptions in AI code
| Exception | Cause |
|-----------|-------|
| `FileNotFoundError` | Dataset file doesn't exist |
| `ValueError` | Wrong data type or shape |
| `KeyError` | Missing column in DataFrame |
| `MemoryError` | Dataset too large for RAM |
| `ZeroDivisionError` | Division by zero in normalisation |

In [ ]:
# ── CODE CELL 3 ──────────────────────────────────────────────────────────────
# Exception handling basics

# Example 1: Handling division by zero in normalisation
def normalise(values):
    try:
        min_val = min(values)
        max_val = max(values)
        if max_val == min_val:
            raise ValueError("Cannot normalise: all values are identical!")
        return [(v - min_val) / (max_val - min_val) for v in values]
    except TypeError as e:
        print(f"TypeError: {e} — make sure input is a list of numbers")
        return None
    except ValueError as e:
        print(f"ValueError: {e}")
        return None

print(normalise([10, 20, 30, 40, 50]))
print(normalise([5, 5, 5, 5]))          # all same → ValueError
print(normalise(["a", "b", "c"]))      # wrong type → TypeError
print()

In [ ]:
# ── CODE CELL 4 ──────────────────────────────────────────────────────────────
# Safe file loader — a pattern you will use in every AI project

import pandas as pd

def safe_load_csv(filepath):
    """
    Safely load a CSV file with informative error messages.
    Returns a DataFrame on success, None on failure.
    """
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded '{filepath}': {df.shape[0]} rows, {df.shape[1]} columns")
        return df
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        print("Tip: Check the path and make sure the file was uploaded.")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: File '{filepath}' is empty.")
        return None
    except Exception as e:
        print(f"Unexpected error loading '{filepath}': {e}")
        return None
    finally:
        print("File loading attempt complete.")

# Test with valid and invalid paths
df1 = safe_load_csv("/content/students.csv")        # saved in Lecture 3
print()
df2 = safe_load_csv("/content/nonexistent.csv")     # does not exist

In [ ]:
# ── CODE CELL 5 ──────────────────────────────────────────────────────────────
# Custom exceptions — for professional AI code

class DataValidationError(Exception):
    """Raised when the dataset does not meet requirements."""
    pass

class ModelNotTrainedError(Exception):
    """Raised when predict() is called before fit()."""
    pass


def validate_dataset(df, required_columns, min_rows=10):
    """Check that a DataFrame meets the minimum requirements."""
    missing_cols = [c for c in required_columns if c not in df.columns]
    if missing_cols:
        raise DataValidationError(f"Missing columns: {missing_cols}")
    if len(df) < min_rows:
        raise DataValidationError(f"Dataset has only {len(df)} rows — need at least {min_rows}")
    if df.isnull().any().any():
        n_missing = df.isnull().sum().sum()
        raise DataValidationError(f"Dataset has {n_missing} missing values — clean it first")
    print("Dataset validation passed!")


# Test
import pandas as pd
sample_df = pd.DataFrame({"study_hours": [6,8,4,7,5,9,3,8,7,6],
                           "marks":       [85,92,65,88,72,96,60,90,83,78]})

try:
    validate_dataset(sample_df, required_columns=["study_hours", "marks", "age"])
except DataValidationError as e:
    print(f"Validation failed: {e}")

try:
    validate_dataset(sample_df, required_columns=["study_hours", "marks"])
except DataValidationError as e:
    print(f"Validation failed: {e}")

## 3. Mini Project — End-to-End Student Data Analysis Pipeline

This project combines **everything from Module 2**:

```
OOP (Lecture 1)  +  NumPy (Lecture 2)  +  Pandas (Lecture 3)
+  Matplotlib/Seaborn (Lecture 4)  +  File Handling / Exceptions (Lecture 5)
```

**Goal:** Build a reusable `DataPipeline` class that:
1. Loads a CSV dataset safely
2. Validates and cleans the data
3. Analyses and summarises it
4. Generates a visualisation report
5. Saves results to files

In [ ]:
# ── MINI PROJECT ─────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

plt.style.use("seaborn-v0_8-whitegrid")


class DataPipeline:
    """
    A reusable data analysis pipeline for AI projects.
    Demonstrates: OOP, NumPy, Pandas, Matplotlib, File Handling, Exceptions.
    """

    def __init__(self, name):
        self.name    = name
        self.df      = None
        self.report  = {}
        self.output_dir = "/content/pipeline_output"
        os.makedirs(self.output_dir, exist_ok=True)

    # ── 1. Load ───────────────────────────────────────────────────────────────
    def load(self, filepath_or_df):
        """Load data from a CSV path or pass a DataFrame directly."""
        try:
            if isinstance(filepath_or_df, pd.DataFrame):
                self.df = filepath_or_df.copy()
            else:
                self.df = pd.read_csv(filepath_or_df)
            print(f"[LOAD] Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
        except FileNotFoundError:
            print(f"[ERROR] File not found: {filepath_or_df}")
        except Exception as e:
            print(f"[ERROR] {e}")
        return self

    # ── 2. Clean ──────────────────────────────────────────────────────────────
    def clean(self):
        """Remove duplicates and fill missing values."""
        if self.df is None:
            print("[ERROR] Load data first!"); return self

        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        after  = len(self.df)
        print(f"[CLEAN] Removed {before - after} duplicate rows")

        missing = self.df.isnull().sum().sum()
        # Fill numeric NaN with column median
        for col in self.df.select_dtypes(include=[np.number]).columns:
            self.df[col].fillna(self.df[col].median(), inplace=True)
        # Fill text NaN with 'Unknown'
        self.df.fillna("Unknown", inplace=True)
        print(f"[CLEAN] Filled {missing} missing values")
        return self

    # ── 3. Analyse ────────────────────────────────────────────────────────────
    def analyse(self, target_col):
        """Compute summary statistics and store in self.report."""
        if self.df is None:
            print("[ERROR] Load data first!"); return self
        if target_col not in self.df.columns:
            print(f"[ERROR] Column '{target_col}' not found"); return self

        target = self.df[target_col]
        self.report = {
            "dataset":    self.name,
            "rows":       len(self.df),
            "columns":    list(self.df.columns),
            "target":     target_col,
            "mean":       round(target.mean(), 2),
            "median":     round(target.median(), 2),
            "std":        round(target.std(), 2),
            "min":        round(target.min(), 2),
            "max":        round(target.max(), 2),
            "pass_rate":  f"{(target >= 60).mean()*100:.1f}%"
        }

        print("[ANALYSE] Summary Report")
        print("=" * 40)
        for k, v in self.report.items():
            if k not in ("columns",):
                print(f"  {k:<14}: {v}")
        return self

    # ── 4. Visualise ──────────────────────────────────────────────────────────
    def visualise(self, target_col, feature_col):
        """Generate a 2×2 analysis chart and save it."""
        if self.df is None:
            print("[ERROR] Load data first!"); return self

        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        fig.suptitle(f"Data Analysis Report — {self.name}", fontsize=14, fontweight="bold")

        # 1. Histogram of target
        axes[0,0].hist(self.df[target_col], bins=15,
                       color="steelblue", edgecolor="white")
        axes[0,0].axvline(self.df[target_col].mean(), color="red",
                           linestyle="--", label=f"Mean: {self.df[target_col].mean():.1f}")
        axes[0,0].set_title(f"Distribution of {target_col}")
        axes[0,0].legend()

        # 2. Scatter: feature vs target
        axes[0,1].scatter(self.df[feature_col], self.df[target_col],
                           alpha=0.6, color="seagreen", s=50)
        m, b = np.polyfit(self.df[feature_col], self.df[target_col], 1)
        x_range = np.linspace(self.df[feature_col].min(), self.df[feature_col].max(), 100)
        axes[0,1].plot(x_range, m*x_range + b, color="red", linewidth=2)
        axes[0,1].set_xlabel(feature_col)
        axes[0,1].set_ylabel(target_col)
        axes[0,1].set_title(f"{feature_col} vs {target_col}")

        # 3. Correlation heatmap
        numeric = self.df.select_dtypes(include=[np.number])
        sns.heatmap(numeric.corr(), annot=True, fmt=".2f",
                    cmap="coolwarm", center=0, ax=axes[1,0], square=True)
        axes[1,0].set_title("Correlation Heatmap")

        # 4. Box plots of numeric features
        numeric.boxplot(ax=axes[1,1])
        axes[1,1].set_title("Box Plots — All Numeric Features")
        axes[1,1].tick_params(axis='x', rotation=15)

        plt.tight_layout()
        chart_path = f"{self.output_dir}/analysis_report.png"
        plt.savefig(chart_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[VISUALISE] Chart saved to {chart_path}")
        return self

    # ── 5. Save ───────────────────────────────────────────────────────────────
    def save(self):
        """Save the cleaned dataset and report to files."""
        if self.df is None:
            print("[ERROR] Nothing to save!"); return self

        csv_path  = f"{self.output_dir}/cleaned_data.csv"
        json_path = f"{self.output_dir}/report.json"

        self.df.to_csv(csv_path, index=False)
        with open(json_path, "w") as f:
            json.dump(self.report, f, indent=4)

        print(f"[SAVE] Cleaned data → {csv_path}")
        print(f"[SAVE] Report       → {json_path}")
        return self

    def __str__(self):
        rows = self.df.shape[0] if self.df is not None else 0
        return f"DataPipeline(name='{self.name}', rows={rows})"


# ── Generate sample dataset ───────────────────────────────────────────────────
np.random.seed(0)
n = 80
sample = pd.DataFrame({
    "study_hours":  np.random.uniform(1, 12, n).round(1),
    "sleep_hours":  np.random.uniform(4, 9,  n).round(1),
    "age":          np.random.randint(18, 30, n).astype(float),
})
sample["marks"] = (sample["study_hours"]*6 + sample["sleep_hours"]*2
                   + np.random.normal(0, 5, n)).clip(30, 100).round(1)
# Introduce some messiness
sample.loc[np.random.choice(n, 5), "marks"] = np.nan
sample = pd.concat([sample, sample.iloc[:3]], ignore_index=True)  # duplicates

print(f"Generated sample dataset: {sample.shape}")

# ── Run the full pipeline ─────────────────────────────────────────────────────
print("\n" + "="*55)
print(" RUNNING FULL DATA PIPELINE")
print("="*55 + "\n")

pipeline = DataPipeline("Student Performance Dataset")
(
    pipeline
    .load(sample)
    .clean()
    .analyse("marks")
    .visualise("marks", "study_hours")
    .save()
)

print(f"\nPipeline: {pipeline}")

## 4. Module 2 — Complete Recap

| Lecture | Tool | What You Can Now Do |
|---------|------|--------------------|
| 1 | OOP | Build classes, use inheritance, read AI library code |
| 2 | NumPy | Fast arrays, matrix math, vectorised ops — AI math foundation |
| 3 | Pandas | Load, clean, filter, group, and prepare real datasets |
| 4 | Matplotlib/Seaborn | Visualise data — distributions, correlations, trends |
| 5 | Files + Exceptions | Write robust, crash-proof, production-quality code |

In [ ]:
# ── MODULE 2 COMPLETION CHECKLIST ────────────────────────────────────────────
# Update True/False values honestly

checklist = [
    ("Completed the OOP Classroom practice task",         False),
    ("Completed all 3 NumPy practice tasks",              False),
    ("Loaded and explored the Titanic dataset with Pandas",False),
    ("Created a 2×2 Matplotlib subplot figure",           False),
    ("Run the full DataPipeline mini project",            False),
    ("Uploaded all 5 notebooks to GitHub",                False),
]

done  = sum(1 for _, s in checklist if s)
total = len(checklist)

print("MODULE 2 COMPLETION CHECKLIST")
print("=" * 50)
for task, status in checklist:
    print(f"  {'[x]' if status else '[ ]'}  {task}")
print(f"\nProgress: {done}/{total} ({done/total*100:.0f}%)")
if done == total:
    print("Module 2 complete — ready for Machine Learning!")
else:
    print(f"{total - done} task(s) remaining before Module 3.")

## What is Coming in Module 3?

**Module 3: Machine Learning** — Week 3

```
Lecture 1: ML fundamentals — supervised vs unsupervised, train/test split
Lecture 2: Regression — Linear Regression, predicting continuous values
Lecture 3: Classification — Logistic Regression, Decision Trees, SVM
Lecture 4: Ensemble methods — Random Forest, XGBoost
Lecture 5: Model evaluation + Mini Project — predicting student marks
```

---

## Module 2 Final Homework
1. Extend the `DataPipeline` class with a `split()` method that splits data into 80% train / 20% test
2. Upload all 5 Module 2 notebooks to your GitHub repository
3. Find a dataset on Kaggle, run it through the pipeline, and push the notebook to GitHub

---
> **Remember:** Python is not a goal — it is your tool. Every line you write in Module 2 will be used again in Modules 3 through 10. The stronger your Python foundation, the faster you build AI.

---
**Next Module:** Machine Learning — teaching machines to learn from data using Scikit-learn